# Collect Results into the Report

Read the results of every run on one dataset (`aircheck/output/<DATASET>/`) and write the report tables as
`aircheck/report/report-<DATASET>.md`: data, setup, cross-validation results, final test results and run details.

Run it after the steps of the [usage recipe](../USAGE_RECIPE.md). It finds each run by the recipe's default names
(editable below); a run that has not been done yet shows as "—". Rerun it any time to refresh the report.
The Summary and Observations sections are left for you to write; rerunning the notebook overwrites the file, so
write them once the numbers are final (or in a copy).

**Environment:** run with the `biomed-multiview` kernel.

## 1. Inputs

In [1]:
from pathlib import Path

# Dataset whose runs to collect (the DATASET used in the other notebooks)
DATASET = "sample-data"

# Train and test files, for the Data section
TRAIN_PATH = Path("~/repos/biomed-multi-view/aircheck/sample_data/train.parquet")
TEST_PATH = Path("~/repos/biomed-multi-view/aircheck/sample_data/test.parquet")
SMILES_COL = "SMILES"
LABEL_COL = "LABEL"

# Run names (the defaults used in the recipe)
MMELON_CV = "mmelon-cv"                    # MMELON_finetuning_cv RUN_NAME
SINGLEVIEW_CV = "singleview-cv"            # MMELON_singleview_finetuning_cv RUN_NAME (folders <name>_<VIEW>)
MMELON_FINAL = "mmelon-final"              # MMELON_finetuning RUN_NAME (latest run is used)
SINGLEVIEW_FINAL = "singleview-final"      # MMELON_singleview_finetuning RUN_NAME (latest run per VIEW is used)
LGBM_PRETRAINED = "lgbm-{view}-pretrained" # MMELON_embedding_lgbm_cv RUN_NAME on pretrained embeddings
LGBM_FINETUNED = "lgbm-{view}-finetuned"   # ... and on fine-tuned embeddings
PREDICTION_NAME = "test"                   # PREDICTION_NAME used in the inference notebooks

REPORT_PATH = Path("../report") / f"report-{DATASET}.md"

## 2. Collect

In [2]:
import json
import re
from datetime import datetime

import numpy as np
import pandas as pd

import pyarrow.parquet as pq


def read_table(path, columns, optional=()) -> pd.DataFrame:
    # Read only the needed columns from a .csv or .parquet file; optional columns are read if present
    path = Path(path)
    is_parquet = path.suffix.lower() in (".parquet", ".pq")
    available = pq.ParquetFile(path).schema_arrow.names if is_parquet else pd.read_csv(path, nrows=0).columns.tolist()
    missing = [c for c in columns if c not in available]
    assert not missing, f"{path.name} has no column(s) {missing}; available: {available}"
    wanted = list(dict.fromkeys([*columns, *(c for c in optional if c in available)]))
    df = pd.read_parquet(path, columns=wanted) if is_parquet else pd.read_csv(path, usecols=wanted)
    return df[wanted].reset_index(drop=True)



def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "bmfm_sm").is_dir() and (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError(f"Run this notebook from inside the biomed-multi-view repo (cwd={start})")


OUT = find_repo_root(Path.cwd().resolve()) / "aircheck" / "output" / DATASET
assert OUT.is_dir(), f"No results for DATASET {DATASET!r}: {OUT} does not exist"

VIEWS = ["text", "image", "graph"]
LGBM_VIEWS = ["fused", "text", "image", "graph"]
COLUMNS = [("hit@200", "hit@200"), ("hit@100", "hit@100"), ("hit@50", "hit@50"), ("rocauc", "AUROC"), ("pr_auc", "PR AUC")]
DASH = "—"


def read_json(path: Path):
    return json.loads(path.read_text()) if path.exists() else None


def latest(pattern: str):
    runs = sorted((OUT / "finetuned").glob(pattern)) if (OUT / "finetuned").is_dir() else []
    return runs[-1] if runs else None


def fmt(value, metric):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return DASH
    if metric.startswith("hit@"):
        return f"{value:.0f}" if float(value).is_integer() else f"{value:.1f}"
    return f"{value:.3f}"


def fmt_cv(mean, std, metric):
    base = fmt(mean, metric)
    if base == DASH or std is None or (isinstance(std, float) and np.isnan(std)):
        return base
    return f"{base} ± {fmt(std, metric)}"


def table(header, rows):
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    return "\n".join(lines + ["| " + " | ".join(str(c) for c in r) + " |" for r in rows])


def duration(folder: Path):
    # Run time: from progress.log timestamps when present, else from file modification times
    if folder is None or not folder.exists():
        return DASH
    log = folder / "progress.log"
    if log.exists():
        stamps = re.findall(r"^(\d{4}-\d\d-\d\d \d\d:\d\d:\d\d)", log.read_text(), flags=re.M)
        if len(stamps) >= 2:
            seconds = (datetime.fromisoformat(stamps[-1]) - datetime.fromisoformat(stamps[0])).total_seconds()
            return f"{seconds / 3600:.1f} h" if seconds >= 3600 else f"{seconds / 60:.0f} min"
    files = [p for p in folder.rglob("*") if p.is_file() and "predictions" not in p.relative_to(folder).parts]
    if len(files) < 2:
        return DASH
    seconds = max(p.stat().st_mtime for p in files) - min(p.stat().st_mtime for p in files)
    return f"≈ {seconds / 3600:.1f} h" if seconds >= 3600 else f"≈ {max(seconds, 60) / 60:.0f} min"


print(f"Collecting results from {OUT}")

In [3]:
# ── Data ──────────────────────────────────────────────────────────────────────
data_rows, sizes = [], {}
for name, path in (("Train", TRAIN_PATH), ("Test", TEST_PATH)):
    df = read_table(path, [SMILES_COL, LABEL_COL])
    sizes[name] = df
    data_rows.append((name, f"`{Path(path).name}`", f"{len(df):,}", f"{int(df[LABEL_COL].sum()):,}",
                      f"{100 * df[LABEL_COL].mean():.2f}%"))
overlap = len(set(sizes["Train"][SMILES_COL]) & set(sizes["Test"][SMILES_COL]))
test_rate = sizes["Test"][LABEL_COL].mean()

folds_npz = OUT / "cv_folds" / "train_cluster_cv_folds.npz"
if folds_npz.exists():
    z = np.load(folds_npz)
    folds_csv = read_table(OUT / "cv_folds" / "train_cluster_cv_data.csv", [LABEL_COL])
    n_folds = len([k for k in z.files if k.endswith("_train")])
    test_pos = [int(folds_csv[LABEL_COL].values[z[f"fold{i}_test"]].sum()) for i in range(n_folds)]
    folds_text = f"cluster-held-out, {n_folds} fold(s); test actives per fold: {', '.join(map(str, test_pos))}"
else:
    folds_text = "not found (run cluster_cv_folds.ipynb)"

data_md = (table(["", "File", "Molecules", "Actives", "Active rate"], data_rows)
           + f"\n\n- **Folds:** {folds_text}.\n- **Molecules in both train and test:** {overlap:,}.")
print(data_md)

|  | File | Molecules | Actives | Active rate |
|---|---|---|---|---|
| Train | `train.parquet` | 10,221 | 221 | 2.16% |
| Test | `test.parquet` | 1,031 | 31 | 3.01% |

- **Folds:** cluster-held-out, 1 fold(s); test actives per fold: 100.
- **Molecules in both train and test:** 0.


In [4]:
# ── Setup ─────────────────────────────────────────────────────────────────────
mm_cfg = read_json(OUT / MMELON_CV / "config.json") or {}
lgbm_choices = sorted({c["lgbm_params_choice"] for c in
                       (read_json(OUT / "lgbm" / LGBM_PRETRAINED.format(view=v) / "config.json") for v in LGBM_VIEWS)
                       if c and "lgbm_params_choice" in c} |
                      {c["lgbm_params_choice"] for c in
                       (read_json(OUT / "lgbm" / LGBM_FINETUNED.format(view=v) / "config.json") for v in LGBM_VIEWS)
                       if c and "lgbm_params_choice" in c})
try:
    import torch
    hardware = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
except Exception:
    hardware = DASH
setup_md = table(["Setting", "Value"], [
    ("Base model", f"`{mm_cfg.get('base_model', 'ibm-research/biomed.sm.mv-te-84m')}`"),
    ("Epochs (maximum, patience)", f"{mm_cfg.get('epochs', DASH)}, {mm_cfg.get('patience', DASH)}"),
    ("Batch size, learning rate", f"{mm_cfg.get('batch_size', DASH)}, {mm_cfg.get('learning_rate', DASH)}"),
    ("Epoch selection", "`SELECT_METRICS`: " + ", ".join(mm_cfg.get("select_metrics", [])) if mm_cfg else DASH),
    ("LightGBM hyperparameters", ", ".join(lgbm_choices) or DASH),
    ("Hardware (where this report was collected)", hardware),
])
print(setup_md)

| Setting | Value |
|---|---|
| Base model | `ibm-research/biomed.sm.mv-te-84m` |
| Epochs (maximum, patience) | 50, 5 |
| Batch size, learning rate | 16, 2e-05 |
| Epoch selection | `SELECT_METRICS`: hit@200, hit@100, hit@50, rocauc |
| LightGBM hyperparameters | default |
| Hardware (where this report was collected) | Tesla T4 |


In [5]:
# ── Cross-validation results ──────────────────────────────────────────────────
def cv_row(label, folder):
    sel = read_json(folder / "selection.json") if folder else None
    if not sel:
        return (label, DASH, *[DASH] * len(COLUMNS))
    return (label, sel["best_epoch"], *[fmt_cv(sel["mean"].get(m), sel["std"].get(m), m) for m, _ in COLUMNS])


def lgbm_cv_row(label, folder):
    path = folder / "cv_summary.csv"
    if not path.exists():
        return (label, "n/a", *[DASH] * len(COLUMNS))
    s = pd.read_csv(path).set_index("metric")
    return (label, "n/a", *[fmt_cv(s["mean"].get(m), s["std"].get(m), m) for m, _ in COLUMNS])


cv_rows = [cv_row("MMELON (multi-view)", OUT / MMELON_CV)]
cv_rows += [cv_row(f"Single view: {v}", OUT / f"{SINGLEVIEW_CV}_{v}") for v in VIEWS]
cv_rows += [lgbm_cv_row(f"LightGBM: {v} embeddings (pretrained)", OUT / "lgbm" / LGBM_PRETRAINED.format(view=v))
            for v in LGBM_VIEWS]
cv_md = table(["Model", "Selected epoch", *[h for _, h in COLUMNS]], cv_rows)
cv_md += ("\n\nMean ± std over folds (no std with one fold). MMELON and single-view CV do not compute PR AUC "
          "during training, so it shows as —.")
print(cv_md)

| Model | Selected epoch | hit@200 | hit@100 | hit@50 | AUROC | PR AUC |
|---|---|---|---|---|---|---|
| MMELON (multi-view) | 5 | 43 | 29 | 19 | 0.857 | — |
| Single view: text | 5 | 45 | 21 | 11 | 0.858 | — |
| Single view: image | 2 | 40 | 22 | 10 | 0.812 | — |
| Single view: graph | 1 | 41 | 22 | 10 | 0.776 | — |
| LightGBM: fused embeddings (pretrained) | n/a | 19 | 10 | 5 | 0.672 | 0.077 |
| LightGBM: text embeddings (pretrained) | n/a | 20 | 11 | 5 | 0.663 | 0.076 |
| LightGBM: image embeddings (pretrained) | n/a | 8 | 5 | 3 | 0.522 | 0.052 |
| LightGBM: graph embeddings (pretrained) | n/a | 20 | 4 | 1 | 0.574 | 0.055 |

Mean ± std over folds (no std with one fold). MMELON and single-view CV do not compute PR AUC during training, so it shows as —.


In [6]:
# ── Final test results ────────────────────────────────────────────────────────
def ensemble_row(label, folder):
    path = folder / "predictions" / PREDICTION_NAME / "metrics.csv"
    if not path.exists():
        return (label, "fold ensemble", *[DASH] * len(COLUMNS))
    m = pd.read_csv(path).set_index("model").loc["ensemble"]
    return (label, "fold ensemble", *[fmt(m.get(c), c) for c, _ in COLUMNS])


def final_row(label, folder):
    metrics = read_json(folder / "predictions" / PREDICTION_NAME / "metrics.json") if folder else None
    if not metrics:
        return (label, "single final model", *[DASH] * len(COLUMNS))
    return (label, "single final model", *[fmt(metrics.get(c), c) for c, _ in COLUMNS])


test_rows = [ensemble_row("MMELON (multi-view)", OUT / MMELON_CV),
             final_row("MMELON (multi-view)", latest(f"{MMELON_FINAL}_[0-9]*"))]
for v in VIEWS:
    test_rows += [ensemble_row(f"Single view: {v}", OUT / f"{SINGLEVIEW_CV}_{v}"),
                  final_row(f"Single view: {v}", latest(f"{SINGLEVIEW_FINAL}_{v}_[0-9]*"))]
test_rows += [ensemble_row(f"LightGBM: {v} embeddings (fine-tuned)", OUT / "lgbm" / LGBM_FINETUNED.format(view=v))
              for v in LGBM_VIEWS]
n_test = len(sizes["Test"])
random_ref = ", ".join(f"hit@{k} ≈ {min(k, n_test) * test_rate:.1f}" for k in (200, 100, 50))
test_md = table(["Model", "Scored with", *[h for _, h in COLUMNS]], test_rows)
test_md += f"\n\nFor reference, a random ranking of the test set would give {random_ref}."
print(test_md)

| Model | Scored with | hit@200 | hit@100 | hit@50 | AUROC | PR AUC |
|---|---|---|---|---|---|---|
| MMELON (multi-view) | fold ensemble | 20 | 17 | 13 | 0.822 | 0.361 |
| MMELON (multi-view) | single final model | 23 | 13 | 10 | 0.820 | 0.337 |
| Single view: text | fold ensemble | 17 | 13 | 8 | 0.775 | 0.129 |
| Single view: text | single final model | 20 | 14 | 11 | 0.798 | 0.150 |
| Single view: image | fold ensemble | 19 | 12 | 11 | 0.811 | 0.177 |
| Single view: image | single final model | 22 | 20 | 15 | 0.846 | 0.352 |
| Single view: graph | fold ensemble | 17 | 15 | 10 | 0.832 | 0.213 |
| Single view: graph | single final model | 23 | 17 | 13 | 0.848 | 0.360 |
| LightGBM: fused embeddings (fine-tuned) | fold ensemble | 16 | 10 | 5 | 0.655 | 0.137 |
| LightGBM: text embeddings (fine-tuned) | fold ensemble | 15 | 10 | 6 | 0.695 | 0.114 |
| LightGBM: image embeddings (fine-tuned) | fold ensemble | 15 | 14 | 11 | 0.651 | 0.317 |
| LightGBM: graph embeddings (fine-tuned) | fold en

In [7]:
# ── Run details ───────────────────────────────────────────────────────────────
detail_rows = [("Folds", OUT / "cv_folds"), ("MMELON CV", OUT / MMELON_CV)]
detail_rows += [(f"Single view CV: {v}", OUT / f"{SINGLEVIEW_CV}_{v}") for v in VIEWS]
detail_rows += [("MMELON final model", latest(f"{MMELON_FINAL}_[0-9]*"))]
detail_rows += [(f"Single view final model: {v}", latest(f"{SINGLEVIEW_FINAL}_{v}_[0-9]*")) for v in VIEWS]
detail_rows += [(f"Embeddings: {p.name}", p) for p in sorted((OUT / "embeddings").glob("*"))] if (OUT / "embeddings").is_dir() else []
detail_rows += [(f"LightGBM: {p.name}", p) for p in sorted((OUT / "lgbm").glob("*"))] if (OUT / "lgbm").is_dir() else []
details_md = table(["Step", "Run folder", "Run time"], [
    (step, f"`{folder.relative_to(OUT.parent.parent)}`" if folder and folder.exists() else DASH,
     duration(folder) if folder and folder.exists() else DASH)
    for step, folder in detail_rows
])
print(details_md)

| Step | Run folder | Run time |
|---|---|---|
| Folds | `output/sample-data/cv_folds` | ≈ 1 min |
| MMELON CV | `output/sample-data/mmelon-cv` | 33 min |
| Single view CV: text | `output/sample-data/singleview-cv_text` | 12 min |
| Single view CV: image | `output/sample-data/singleview-cv_image` | 5 min |
| Single view CV: graph | `output/sample-data/singleview-cv_graph` | 9 min |
| MMELON final model | `output/sample-data/finetuned/mmelon-final_20261020_162634` | ≈ 21 min |
| Single view final model: text | `output/sample-data/finetuned/singleview-final_text_20261020_164954` | ≈ 8 min |
| Single view final model: image | `output/sample-data/finetuned/singleview-final_image_20261020_165945` | ≈ 3 min |
| Single view final model: graph | `output/sample-data/finetuned/singleview-final_graph_20261020_170415` | ≈ 3 min |
| Embeddings: test-finetuned | `output/sample-data/embeddings/test-finetuned` | ≈ 1 min |
| Embeddings: test-pretrained | `output/sample-data/embeddings/test-pretrained` 

## 3. Write the report

In [8]:
report = f"""# Report: {DATASET}

*Tables generated by `collect_results.ipynb` on {datetime.now():%Y-%m-%d %H:%M} from `aircheck/output/{DATASET}/`.
Rerunning the notebook regenerates this file.*

## Summary

_To write: which model ranks actives best on the final test set (hit@K), and how CV and test results compare._

## Data

{data_md}

## Setup

{setup_md}

## Cross-validation results (train)

{cv_md}

## Final test results: fine-tuned models

All models here were fine-tuned on the train set. LightGBM rows use embeddings from the fine-tuned MMELON model
(the single final model) and are scored as a fold ensemble.

{test_md}

## Observations

_To write:_
- Best model on the final test set, and by how much.
- Multi-view versus single views: does fusing the views help?
- How close the CV estimates are to the test results.

## Caveats

- Cluster-held-out CV tests each fold on unseen chemistry, so it is expected to score lower than random splits.
- The selected epoch is chosen on the CV test folds, so the CV numbers are slightly optimistic.
- If LightGBM used embeddings from a fine-tuned model, its CV numbers are over-optimistic (see the recipe); only its
  test numbers count.

## Run details

{details_md}
"""
REPORT_PATH = Path(REPORT_PATH)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(report)
print(f"Wrote {REPORT_PATH.resolve()}")

from IPython.display import Markdown
display(Markdown(report))

Wrote ~/repos/biomed-multi-view/aircheck/report/report-sample-data.md


# Report: sample-data

*Tables generated by `collect_results.ipynb` on 2026-10-20 19:45 from `aircheck/output/sample-data/`.
Rerunning the notebook regenerates this file.*

## Summary

_To write: which model ranks actives best on the final test set (hit@K), and how CV and test results compare._

## Data

|  | File | Molecules | Actives | Active rate |
|---|---|---|---|---|
| Train | `train.parquet` | 10,221 | 221 | 2.16% |
| Test | `test.parquet` | 1,031 | 31 | 3.01% |

- **Folds:** cluster-held-out, 1 fold(s); test actives per fold: 100.
- **Molecules in both train and test:** 0.

## Setup

| Setting | Value |
|---|---|
| Base model | `ibm-research/biomed.sm.mv-te-84m` |
| Epochs (maximum, patience) | 50, 5 |
| Batch size, learning rate | 16, 2e-05 |
| Epoch selection | `SELECT_METRICS`: hit@200, hit@100, hit@50, rocauc |
| LightGBM hyperparameters | default |
| Hardware (where this report was collected) | Tesla T4 |

## Cross-validation results (train)

| Model | Selected epoch | hit@200 | hit@100 | hit@50 | AUROC | PR AUC |
|---|---|---|---|---|---|---|
| MMELON (multi-view) | 5 | 43 | 29 | 19 | 0.857 | — |
| Single view: text | 5 | 45 | 21 | 11 | 0.858 | — |
| Single view: image | 2 | 40 | 22 | 10 | 0.812 | — |
| Single view: graph | 1 | 41 | 22 | 10 | 0.776 | — |
| LightGBM: fused embeddings (pretrained) | n/a | 19 | 10 | 5 | 0.672 | 0.077 |
| LightGBM: text embeddings (pretrained) | n/a | 20 | 11 | 5 | 0.663 | 0.076 |
| LightGBM: image embeddings (pretrained) | n/a | 8 | 5 | 3 | 0.522 | 0.052 |
| LightGBM: graph embeddings (pretrained) | n/a | 20 | 4 | 1 | 0.574 | 0.055 |

Mean ± std over folds (no std with one fold). MMELON and single-view CV do not compute PR AUC during training, so it shows as —.

## Final test results: fine-tuned models

All models here were fine-tuned on the train set. LightGBM rows use embeddings from the fine-tuned MMELON model
(the single final model) and are scored as a fold ensemble.

| Model | Scored with | hit@200 | hit@100 | hit@50 | AUROC | PR AUC |
|---|---|---|---|---|---|---|
| MMELON (multi-view) | fold ensemble | 20 | 17 | 13 | 0.822 | 0.361 |
| MMELON (multi-view) | single final model | 23 | 13 | 10 | 0.820 | 0.337 |
| Single view: text | fold ensemble | 17 | 13 | 8 | 0.775 | 0.129 |
| Single view: text | single final model | 20 | 14 | 11 | 0.798 | 0.150 |
| Single view: image | fold ensemble | 19 | 12 | 11 | 0.811 | 0.177 |
| Single view: image | single final model | 22 | 20 | 15 | 0.846 | 0.352 |
| Single view: graph | fold ensemble | 17 | 15 | 10 | 0.832 | 0.213 |
| Single view: graph | single final model | 23 | 17 | 13 | 0.848 | 0.360 |
| LightGBM: fused embeddings (fine-tuned) | fold ensemble | 16 | 10 | 5 | 0.655 | 0.137 |
| LightGBM: text embeddings (fine-tuned) | fold ensemble | 15 | 10 | 6 | 0.695 | 0.114 |
| LightGBM: image embeddings (fine-tuned) | fold ensemble | 15 | 14 | 11 | 0.651 | 0.317 |
| LightGBM: graph embeddings (fine-tuned) | fold ensemble | 10 | 7 | 4 | 0.642 | 0.052 |

For reference, a random ranking of the test set would give hit@200 ≈ 6.0, hit@100 ≈ 3.0, hit@50 ≈ 1.5.

## Observations

_To write:_
- Best model on the final test set, and by how much.
- Multi-view versus single views: does fusing the views help?
- How close the CV estimates are to the test results.

## Caveats

- Cluster-held-out CV tests each fold on unseen chemistry, so it is expected to score lower than random splits.
- The selected epoch is chosen on the CV test folds, so the CV numbers are slightly optimistic.
- If LightGBM used embeddings from a fine-tuned model, its CV numbers are over-optimistic (see the recipe); only its
  test numbers count.

## Run details

| Step | Run folder | Run time |
|---|---|---|
| Folds | `output/sample-data/cv_folds` | ≈ 1 min |
| MMELON CV | `output/sample-data/mmelon-cv` | 33 min |
| Single view CV: text | `output/sample-data/singleview-cv_text` | 12 min |
| Single view CV: image | `output/sample-data/singleview-cv_image` | 5 min |
| Single view CV: graph | `output/sample-data/singleview-cv_graph` | 9 min |
| MMELON final model | `output/sample-data/finetuned/mmelon-final_20261020_162634` | ≈ 21 min |
| Single view final model: text | `output/sample-data/finetuned/singleview-final_text_20261020_164954` | ≈ 8 min |
| Single view final model: image | `output/sample-data/finetuned/singleview-final_image_20261020_165945` | ≈ 3 min |
| Single view final model: graph | `output/sample-data/finetuned/singleview-final_graph_20261020_170415` | ≈ 3 min |
| Embeddings: test-finetuned | `output/sample-data/embeddings/test-finetuned` | ≈ 1 min |
| Embeddings: test-pretrained | `output/sample-data/embeddings/test-pretrained` | ≈ 1 min |
| Embeddings: train-finetuned | `output/sample-data/embeddings/train-finetuned` | ≈ 2 min |
| Embeddings: train-pretrained | `output/sample-data/embeddings/train-pretrained` | ≈ 2 min |
| LightGBM: lgbm-fused-finetuned | `output/sample-data/lgbm/lgbm-fused-finetuned` | ≈ 1 min |
| LightGBM: lgbm-fused-pretrained | `output/sample-data/lgbm/lgbm-fused-pretrained` | ≈ 1 min |
| LightGBM: lgbm-graph-finetuned | `output/sample-data/lgbm/lgbm-graph-finetuned` | ≈ 1 min |
| LightGBM: lgbm-graph-pretrained | `output/sample-data/lgbm/lgbm-graph-pretrained` | ≈ 1 min |
| LightGBM: lgbm-image-finetuned | `output/sample-data/lgbm/lgbm-image-finetuned` | ≈ 1 min |
| LightGBM: lgbm-image-pretrained | `output/sample-data/lgbm/lgbm-image-pretrained` | ≈ 1 min |
| LightGBM: lgbm-text-finetuned | `output/sample-data/lgbm/lgbm-text-finetuned` | ≈ 1 min |
| LightGBM: lgbm-text-pretrained | `output/sample-data/lgbm/lgbm-text-pretrained` | ≈ 1 min |
